In [ ]:
using Pkg
Pkg.activate(".")
Pkg.add(["ControlSystems", "Plots"])

In [ ]:
using ControlSystems
using Plots
using Printf

gr()

In [ ]:
# 入力
wn = 10.0
zetas = [0.2, 0.4, 0.707, 1.0]
t = 0:0.005:2.0

s = tf("s")

# グラフ枠
p1 = plot(title="Step Response", xlabel="Time [s]", ylabel="Amplitude", grid=true)    # ステップ
p2 = plot(title="Impulse Response", xlabel="Time [s]", ylabel="Amplitude", grid=true) # インパルス
p3 = plot(title="Ramp Response", xlabel="Time [s]", ylabel="Amplitude", grid=true)    # ランプ

# 波形
for z in zetas
  # 関数
  G = wn^2 / (s^2 + 2*z*wn*s + wn^2)

  # ステップ
  res = step(G, t)
  plot!(p1, res.t, res.y[:], label="ζ = $z", lw=1.5)

  # インパルス
  res = impulse(G, t)
  plot!(p2, res.t, res.y[:], label="ζ = $z", lw=1.5)

  # ランプ
  res = step(G / s, t)
  plot!(p3, res.t, res.y[:], label="ζ = $z", lw=1.5)
end

## 参考線
hline!(p1, [1.0], ls=:dash, lc=:black, label="")
plot!(p3, t, collect(t), label="Input r(t)=t", ls=:dash, lc=:black, lw=1.5)

## 出力
plot(p1, p2, p3,
     layout = (1, 3),
     size = (1100, 350),
     bottom_margin = 10Plots.mm,
     left_margin = 5Plots.mm)

In [ ]:
function evaluate_performance(G, wn, zeta; t_end=3.0, dt=0.001)
  t = 0:dt:t_end

  # ステップ応答のピーク
  step_res = step(G, t)
  y_step = step_res.y[:]
  dy = diff(y_step)

  pos_peaks = findall(i -> dy[i] > 0 && dy[i+1] <= 0, 1:length(dy)-1)
  neg_peaks = findall(i -> dy[i] < 0 && dy[i+1] >= 0, 1:length(dy)-1)

  ## 正ピーク
  if !isempty(pos_peaks)
    ### 振動あり
    t_peak1 = t[pos_peaks[1] + 1]
    y_peak1 = y_step[pos_peaks[1] + 1]
    overshoot = max(0.0, (y_peak1 - 1.0) * 100)
  else
    ### 振動なし
    t_peak1 = NaN
    y_peak1 = NaN
    overshoot = 0.0
  end
  ## 負ピーク
  if !isempty(neg_peaks)
    ### 振動あり
    t_bottom1 = t[neg_peaks[1] + 1]
    y_bottom1 = y_step[neg_peaks[1] + 1]
  else
    ### 振動なし
    t_bottom1 = NaN
    y_bottom1 = NaN
  end

  # インパルス
  imp_res = impulse(G, t)
  max_imp, idx_imp = findmax(imp_res.y[:])
  t_imp_peak = t[idx_imp]

  # ランプ
  ramp_res = step(G / s, t)
  ramp_lag = t[end] - ramp_res.y[:][end]

  # 理論値
  if zeta < 1.0
    # 振動するパラメータ時
    wd = wn * sqrt(1.0 - zeta^2)
    th_peak1 = π / wd
    th_bottom1 = 2π / wd
    th_os = exp(-π * zeta / sqrt(1.0 - zeta^2)) * 100
  else
    #振動しないパラメータ時
    wd = NaN
    th_peak1 = NaN
    th_bottom1 = NaN
    th_os = 0.0
  end
  th_lag = 2 * zeta / wn

  @printf("================== ζ = %.3f, ωn = %.1f rad/s ==================\n", zeta, wn)
  @printf("  【ステップ応答】\n")
  @printf("    オーバーシュート: 実測 %5.1f %%  (理論値: %5.1f %%)\n", overshoot, th_os)
  @printf("    第1山到達時間:   実測 %5.4f s  (理論値: %5.4f s)\n", t_peak1, th_peak1)
  @printf("    第1谷到達時間:   実測 %5.4f s  (理論値: %5.4f s)\n", t_bottom1, th_bottom1)
  @printf("  【インパルス応答】\n")
  @printf("    ピーク値:       実測 %5.3f    (到達時間: %5.4f s)\n", max_imp, t_imp_peak)
  @printf("  【ランプ応答】\n")
  @printf("    定常遅れ偏差:   実測 %5.4f s  (理論値 2ζ/ωn: %5.4f s)\n\n", ramp_lag, th_lag)
end

# 各減衰定数について数値を評価
for z in zetas
  G = wn^2 / (s^2 + 2*z*wn*s + wn^2)
  evaluate_performance(G, wn, z)
end

In [ ]:
# 入力
# 一次遅れ系
T = 1.0 / wn
G1 = tf([1.0], [T, 1.0])
# 二次遅れ系
G2_vibe = tf([wn^2], [1.0, 2*0.3*wn, wn^2])  # 振動的 (ζ=0.3)
G2_flat = tf([wn^2], [1.0, 2*0.707*wn, wn^2]) # 平坦 (ζ=0.707)

# 1. 時間応答の比較
p_step_comp = plot(title="Step: 1st vs 2nd", xlabel="Time [s]", ylabel="y(t)", grid=true)
plot!(p_step_comp, step(G1, t), label="1st-order (T=0.1s)", lw=2)
plot!(p_step_comp, step(G2_vibe, t), label="2nd-order (ζ=0.3)", lw=2)
plot!(p_step_comp, step(G2_flat, t), label="2nd-order (ζ=0.707)", lw=2)
hline!(p_step_comp, [1.0], ls=:dash, lc=:black, label="")

# 2. ボード線図（周波数応答）
w = 10 .^ range(-0.5, 2.5, length=200) # 0.3 rad/s 〜 300 rad/s
p_bode = bodeplot([G1, G2_vibe, G2_flat], w,
                  plotphase=true,
                  label=["1st (T=0.1)" "2nd (ζ=0.3)" "2nd (ζ=0.707)"])

display(p_step_comp)
display(p_bode)

In [ ]:
# プラント（制御対象）：時定数 0.5s、定常ゲイン 1 の一次遅れ系
P = tf([1.0], [0.5, 1.0])

# PI制御器のパラメータ設定
s = tf("s")
Kp = 2.0
Ki_candidates = [0.5, 2.0, 6.0] # 積分ゲインを段階的に増やす

p_pi = plot(title="PI Control: Closed-Loop Step Response",
            xlabel="Time [s]", ylabel="Output y(t)", grid=true)
hline!(p_pi, [1.0], ls=:dash, lc=:black, label="Target r=1.0")

for Ki in Ki_candidates
    C = Kp + Ki/s                # PI制御器
    L = P * C                    # 一巡伝達関数
    T_cl = feedback(L)           # 閉ループ系 L / (1 + L)

    res = step(T_cl, 0:0.01:4.0)
    plot!(p_pi, res.t, res.y[:], label="Kp=$Kp, Ki=$Ki", lw=2)
end

display(p_pi)